# Bubble Bi

### Reading the stock market like a language

## The problem

Every day, a stock leaves a trace: open, high, low, close, volume. Markets repeat certain moods — calm drift, panic, sharp reversal — but nobody labels them, and there is no dictionary of them.

**The idea:** let the machine build that dictionary itself.

We compress each stock-day into a single **token** — one word out of 512 that the machine invents on its own. A stock's history then becomes a sentence:

```
AAPL: ... 147  147  391  208  208  63  →  what comes next?
```

From there it is the same trick as ChatGPT: read the sentence, predict the next word.

**Three stages:**

| | | |
|---|---|---|
| 1. **Tokenizer** | turns each stock-day into one token | ✅ built |
| 2. **Predictor** | a GPT that predicts the next token | ✅ built |
| 3. **Trader** | acts on it: long / short / flat | ⬜ not built |

> ⚠️ **The one rule:** nothing may ever look into the future. Break it and the model looks brilliant while losing real money. A test enforces it.

Run the cells below in order.

---
## 1. Settings

Every knob is here — nothing hidden in a config file.

**The tokenizer looks at each day through two windows**, then merges what it sees into a single token:

```
  TS  — what THIS stock has been doing   (one stock, over time)   ┐
                                                                  ├──→  ONE token
  CS  — what the WHOLE MARKET was doing  (all stocks, on a day)   ┘
```

Each window has its own settings.

In [ ]:
SETTINGS = dict(

    # ── Which companies, and how far back ────────────────────────────────
    tickers = ["AAPL", "MSFT", "AMZN", "GOOGL", "META", "NVDA", "JPM", "V", "JNJ", "WMT",
               "PG", "HD", "BAC", "XOM", "CVX", "KO", "PEP", "DIS", "CSCO", "INTC",
               "VZ", "T", "MRK", "PFE", "ABT", "NKE", "MCD", "CAT", "BA", "IBM"],
    start   = "2010-01-01",

    # ── ENTRY 1 — TS: what THIS stock has been doing ─────────────────────
    ts = dict(
        days          = 4,     # days of the stock's own history it reads
        vocabulary    = 512,   # size of its private dictionary
        encoder_depth = 3,     # how deep it looks
        decoder_depth = 2,     # (only used while training, to check it understood)
    ),

    # ── ENTRY 2 — CS: what the WHOLE MARKET was doing ────────────────────
    cs = dict(
        days          = 5,     # days of market-wide history it reads
        vocabulary    = 512,
        encoder_depth = 3,
        decoder_depth = 2,
    ),

    # ── Where the two entries merge into the ONE token we keep ───────────
    fusion = dict(
        vocabulary = 512,      # THIS is the dictionary the predictor actually reads
        depth      = 2,
    ),

    # ── The predictor: the GPT that reads the sentences ──────────────────
    predictor = dict(
        sentence_length = 64,  # how many past tokens it reads before guessing
        depth           = 4,
    ),

    # ── Shared ───────────────────────────────────────────────────────────
    model_size = 128,   # brain width. Shared on purpose: TS and CS meet in a
                        # cross-attention layer, which needs both the same width.

    # ── Training ─────────────────────────────────────────────────────────
    steps         = 2000,   # how long to train (raise this on a GPU)
    batch_size    = 256,
    learning_rate = 1e-4,
    seed          = 42,     # same seed = same result, every time
)

---
## 2. Get the code

Downloads the project and installs what it needs. Safe to re-run — it skips anything already there.

In [ ]:
import os, subprocess, sys

REPO   = "https://github.com/hockper/Quant-AI-2026.git"
BRANCH = "notebook-first"

if not os.path.exists("bubble_bi"):                  # not here yet -> fetch it
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO, "bubble-bi"], check=True)
    os.chdir("bubble-bi")

sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
               check=True)

import bubble_bi as bb

settings = bb.check(SETTINGS)     # fills in defaults, catches typos
print(bb.summary(settings))

### ✅ Check

Every section of this notebook ends like this: it *proves* the step worked, then tells you what you now have. If something is wrong it stops here rather than letting you carry on.

In [ ]:
bb.verify.setup(settings)